In [1]:
import os
import re
import json
import random
import warnings
import shutil
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# SETTINGS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FAST_MODE = False

if FAST_MODE:
    MAX_ML_SAMPLES = 30000
    SAMPLES_PER_SEQUENCE = 2
else:
    MAX_ML_SAMPLES = 150000
    SAMPLES_PER_SEQUENCE = 5

TOP_N_MODELS = 3
KMER_SIZE = 11
BEAM_WIDTH = 5
RESIDUE_TOP_K = 5

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("==== Settings ====")
print("FAST_MODE:", FAST_MODE)
print("MAX_ML_SAMPLES:", MAX_ML_SAMPLES)
print("KMER_SIZE:", KMER_SIZE)
print("TOP_N_MODELS:", TOP_N_MODELS)
print("BEAM_WIDTH:", BEAM_WIDTH)
print("RESIDUE_TOP_K:", RESIDUE_TOP_K)


# DATASET PATHS

DATA_DIR = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"

DE_NOVO_CAH2 = f"{DATA_DIR}/de_novo_sequence_CAH2.txt"
MAB_TARGET = f"{DATA_DIR}/mab_target_sequence.txt"
MAB_TRAINING = f"{DATA_DIR}/mab_training_sequence.txt"
P5A_TRAINING = f"{DATA_DIR}/p5A_training_sequence.txt"
CAH2_TARGET = f"{DATA_DIR}/target_sequence_CAH2.txt"
CAH2_TRAINING = f"{DATA_DIR}/training_sequences_CAH2.txt"


def find_file(filename, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None


paths = {
    "DE_NOVO_CAH2": ("de_novo_sequence_CAH2.txt", DE_NOVO_CAH2),
    "MAB_TARGET": ("mab_target_sequence.txt", MAB_TARGET),
    "MAB_TRAINING": ("mab_training_sequence.txt", MAB_TRAINING),
    "P5A_TRAINING": ("p5A_training_sequence.txt", P5A_TRAINING),
    "CAH2_TARGET": ("target_sequence_CAH2.txt", CAH2_TARGET),
    "CAH2_TRAINING": ("training_sequences_CAH2.txt", CAH2_TRAINING),
}

fixed_paths = {}

for key, (fname, path) in paths.items():
    if os.path.exists(path):
        fixed_paths[key] = path
    else:
        fixed_paths[key] = find_file(fname)

DE_NOVO_CAH2 = fixed_paths["DE_NOVO_CAH2"]
MAB_TARGET = fixed_paths["MAB_TARGET"]
MAB_TRAINING = fixed_paths["MAB_TRAINING"]
P5A_TRAINING = fixed_paths["P5A_TRAINING"]
CAH2_TARGET = fixed_paths["CAH2_TARGET"]
CAH2_TRAINING = fixed_paths["CAH2_TRAINING"]

print("\n==== Dataset Paths ====")
for name, path in fixed_paths.items():
    exists = path is not None and os.path.exists(path)
    print(name, "=>", path, "| exists:", exists)

required_files = {
    "MAB_TARGET": MAB_TARGET,
    "MAB_TRAINING": MAB_TRAINING,
    "P5A_TRAINING": P5A_TRAINING,
    "CAH2_TARGET": CAH2_TARGET,
    "CAH2_TRAINING": CAH2_TRAINING,
}

for name, path in required_files.items():
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"Required file missing: {name}")


# AMINO ACID VOCABULARY AND MASS TABLE

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
GAP_TOKEN = "-"

AA_MASS = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99,
    "T": 101, "C": 103, "L": 113, "I": 113, "N": 114,
    "D": 115, "Q": 128, "K": 128, "E": 129, "M": 131,
    "H": 137, "F": 147, "R": 156, "Y": 163, "W": 186
}

aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int[GAP_TOKEN] = 0
int_to_aa = {v: k for k, v in aa_to_int.items()}


def peptide_mass(seq):
    return int(sum(AA_MASS.get(aa, 0) for aa in seq))


def is_valid_protein_sequence(seq):
    return all(ch in AMINO_ACIDS for ch in seq)


def encode_sequence(seq):
    return [aa_to_int.get(ch, 0) for ch in seq]


def decode_encoded_kmer(encoded_row):
    return "".join(int_to_aa.get(int(x), GAP_TOKEN) for x in encoded_row)


print("\n==== Amino Acid Checks ====")
print("Amino acids:", AMINO_ACIDS)
print("Mass GPEH:", peptide_mass("GPEH"))
print("Mass SVSYDQA:", peptide_mass("SVSYDQA"))


# FASTA / TEXT PARSER

def read_text(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq


def parse_fasta_or_sequences(path):
    text = read_text(path)
    sequences = []
    current = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current:
                seq = clean_sequence("".join(current))
                if seq:
                    sequences.append(seq)
                current = []
        else:
            current.append(line)

    if current:
        seq = clean_sequence("".join(current))
        if seq:
            sequences.append(seq)

    return sequences



# LOAD AND CURATE SEQUENCES

mab_target_seq = parse_fasta_or_sequences(MAB_TARGET)[0]
mab_train_seqs = parse_fasta_or_sequences(MAB_TRAINING)

p5a_train_seqs = parse_fasta_or_sequences(P5A_TRAINING)

cah2_target_seq = parse_fasta_or_sequences(CAH2_TARGET)[0]
cah2_train_seqs = parse_fasta_or_sequences(CAH2_TRAINING)

if DE_NOVO_CAH2 is not None and os.path.exists(DE_NOVO_CAH2):
    de_novo_cah2_seqs = parse_fasta_or_sequences(DE_NOVO_CAH2)
else:
    de_novo_cah2_seqs = []

all_homologs = mab_train_seqs + p5a_train_seqs + cah2_train_seqs

print("\n==== Loaded Sequences ====")
print("Mab target length:", len(mab_target_seq))
print("Mab training sequences:", len(mab_train_seqs))
print("P5A training sequences:", len(p5a_train_seqs))
print("CAH2 target length:", len(cah2_target_seq))
print("CAH2 training sequences:", len(cah2_train_seqs))
print("De novo CAH2 sequences:", len(de_novo_cah2_seqs))
print("All homolog sequences:", len(all_homologs))


# MASKED K-MER DATASET GENERATION

def generate_masked_kmer_dataset(
    sequences,
    k=11,
    max_samples=150000,
    random_internal_masks=0
):
    """
    Generates masked k-mer samples using:
    - first-position masking
    - middle-position masking
    - last-position masking
    """

    if k % 2 == 0:
        raise ValueError("k must be odd, for example 11, 15, or 21.")

    X = []
    y_letters = []
    middle_pos = k // 2

    for seq in sequences:
        if len(seq) < k:
            continue

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i + k]

            if not is_valid_protein_sequence(kmer):
                continue

            mask_positions = {0, middle_pos, k - 1}

            if random_internal_masks > 0:
                possible_positions = list(range(1, k - 1))
                sampled_positions = random.sample(
                    possible_positions,
                    k=min(random_internal_masks, len(possible_positions))
                )
                mask_positions.update(sampled_positions)

            for pos in mask_positions:
                masked = list(kmer)
                true_aa = masked[pos]
                masked[pos] = GAP_TOKEN

                X.append(encode_sequence("".join(masked)))
                y_letters.append(true_aa)

                if max_samples is not None and len(X) >= max_samples:
                    return np.array(X), np.array(y_letters)

    return np.array(X), np.array(y_letters)


X_ml, y_ml_letters = generate_masked_kmer_dataset(
    all_homologs,
    k=KMER_SIZE,
    max_samples=MAX_ML_SAMPLES,
    random_internal_masks=0
)

label_encoder = LabelEncoder()
y_ml = label_encoder.fit_transform(y_ml_letters)

X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(
    X_ml,
    y_ml,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ml
)

print("\n==== ML Dataset ====")
print("X_ml:", X_ml.shape)
print("y_ml:", y_ml.shape)
print("Classes:", list(label_encoder.classes_))
print("Train:", X_train_ml.shape)
print("Validation:", X_val_ml.shape)


# SAVE GENERATED DATA

GENERATED_DATA_DIR = os.path.join(OUTPUT_DIR, "generated_data")
os.makedirs(GENERATED_DATA_DIR, exist_ok=True)

RAW_CURATED_DIR = os.path.join(GENERATED_DATA_DIR, "01_curated_sequences")
MASKED_DIR = os.path.join(GENERATED_DATA_DIR, "02_masked_11mer_dataset")
SPLIT_DIR = os.path.join(GENERATED_DATA_DIR, "03_train_validation_split")
METADATA_DIR = os.path.join(GENERATED_DATA_DIR, "04_metadata")

for folder in [RAW_CURATED_DIR, MASKED_DIR, SPLIT_DIR, METADATA_DIR]:
    os.makedirs(folder, exist_ok=True)


def save_sequence_list(sequences, output_path, protein_group, prefix):
    rows = []

    for i, seq in enumerate(sequences):
        rows.append({
            "sequence_id": f"{prefix}_{i}",
            "protein_group": protein_group,
            "sequence": seq,
            "length": len(seq)
        })

    pd.DataFrame(rows).to_csv(output_path, index=False)


# Save target sequences
target_df = pd.DataFrame([
    {
        "sequence_id": "MabCampath_target",
        "protein_group": "MabCampath",
        "sequence": mab_target_seq,
        "length": len(mab_target_seq)
    },
    {
        "sequence_id": "CAH2_target",
        "protein_group": "CAH2",
        "sequence": cah2_target_seq,
        "length": len(cah2_target_seq)
    }
])

target_df.to_csv(
    os.path.join(RAW_CURATED_DIR, "target_sequences_cleaned.csv"),
    index=False
)


# Save de novo CAH2 if available
if len(de_novo_cah2_seqs) > 0:
    save_sequence_list(
        de_novo_cah2_seqs,
        os.path.join(RAW_CURATED_DIR, "de_novo_cah2_sequence_cleaned.csv"),
        "CAH2_de_novo",
        "CAH2_DENOVO"
    )


# Save homologous training sequences separately
save_sequence_list(
    mab_train_seqs,
    os.path.join(RAW_CURATED_DIR, "mab_training_sequences_cleaned.csv"),
    "MabCampath",
    "MAB"
)

save_sequence_list(
    p5a_train_seqs,
    os.path.join(RAW_CURATED_DIR, "p5a_training_sequences_cleaned.csv"),
    "P5A",
    "P5A"
)

save_sequence_list(
    cah2_train_seqs,
    os.path.join(RAW_CURATED_DIR, "cah2_training_sequences_cleaned.csv"),
    "CAH2",
    "CAH2"
)


# Save all homologous sequences together
all_homolog_rows = []

for i, seq in enumerate(mab_train_seqs):
    all_homolog_rows.append({
        "sequence_id": f"MAB_{i}",
        "protein_group": "MabCampath",
        "sequence": seq,
        "length": len(seq)
    })

for i, seq in enumerate(p5a_train_seqs):
    all_homolog_rows.append({
        "sequence_id": f"P5A_{i}",
        "protein_group": "P5A",
        "sequence": seq,
        "length": len(seq)
    })

for i, seq in enumerate(cah2_train_seqs):
    all_homolog_rows.append({
        "sequence_id": f"CAH2_{i}",
        "protein_group": "CAH2",
        "sequence": seq,
        "length": len(seq)
    })

all_homologs_df = pd.DataFrame(all_homolog_rows)

all_homologs_df.to_csv(
    os.path.join(RAW_CURATED_DIR, "all_homologous_sequences_cleaned.csv"),
    index=False
)


# Save masked 11-mer dataset
masked_columns = [f"pos_{i+1}" for i in range(KMER_SIZE)]

masked_df = pd.DataFrame(X_ml, columns=masked_columns)

masked_df.insert(
    0,
    "masked_11mer",
    [decode_encoded_kmer(row) for row in X_ml]
)

masked_df["label_amino_acid"] = y_ml_letters
masked_df["label_encoded"] = y_ml

masked_dataset_name = f"masked_{KMER_SIZE}mer_dataset_{X_ml.shape[0]}.csv"

masked_df.to_csv(
    os.path.join(MASKED_DIR, masked_dataset_name),
    index=False
)

masked_df.to_csv(
    os.path.join(MASKED_DIR, masked_dataset_name + ".gz"),
    index=False,
    compression="gzip"
)


# Save train / validation splits
train_df = pd.DataFrame(X_train_ml, columns=masked_columns)

train_df.insert(
    0,
    "masked_11mer",
    [decode_encoded_kmer(row) for row in X_train_ml]
)

train_df["label_amino_acid"] = label_encoder.inverse_transform(y_train_ml)
train_df["label_encoded"] = y_train_ml

val_df = pd.DataFrame(X_val_ml, columns=masked_columns)

val_df.insert(
    0,
    "masked_11mer",
    [decode_encoded_kmer(row) for row in X_val_ml]
)

val_df["label_amino_acid"] = label_encoder.inverse_transform(y_val_ml)
val_df["label_encoded"] = y_val_ml

train_file = f"masked_{KMER_SIZE}mer_train_{X_train_ml.shape[0]}.csv"
val_file = f"masked_{KMER_SIZE}mer_validation_{X_val_ml.shape[0]}.csv"

train_df.to_csv(
    os.path.join(SPLIT_DIR, train_file),
    index=False
)

val_df.to_csv(
    os.path.join(SPLIT_DIR, val_file),
    index=False
)

train_df.to_csv(
    os.path.join(SPLIT_DIR, train_file + ".gz"),
    index=False,
    compression="gzip"
)

val_df.to_csv(
    os.path.join(SPLIT_DIR, val_file + ".gz"),
    index=False,
    compression="gzip"
)


# Save metadata
label_mapping_df = pd.DataFrame({
    "label_encoded": np.arange(len(label_encoder.classes_)),
    "amino_acid": label_encoder.classes_
})

label_mapping_df.to_csv(
    os.path.join(METADATA_DIR, "label_encoding_mapping.csv"),
    index=False
)

aa_encoding_df = pd.DataFrame([
    {
        "amino_acid": aa,
        "integer_encoding": aa_to_int[aa],
        "residue_mass_Da": AA_MASS[aa]
    }
    for aa in AMINO_ACIDS
])

gap_row = pd.DataFrame([{
    "amino_acid": GAP_TOKEN,
    "integer_encoding": 0,
    "residue_mass_Da": 0
}])

aa_encoding_df = pd.concat([aa_encoding_df, gap_row], ignore_index=True)

aa_encoding_df.to_csv(
    os.path.join(METADATA_DIR, "amino_acid_encoding_and_mass_table.csv"),
    index=False
)

settings = {
    "random_state": RANDOM_STATE,
    "fast_mode": FAST_MODE,
    "max_ml_samples": MAX_ML_SAMPLES,
    "samples_per_sequence": SAMPLES_PER_SEQUENCE,
    "kmer_size": KMER_SIZE,
    "top_n_models": TOP_N_MODELS,
    "beam_width": BEAM_WIDTH,
    "residue_top_k": RESIDUE_TOP_K,
    "amino_acid_classes": AMINO_ACIDS,
    "gap_token": GAP_TOKEN,
    "masking_strategy": "first, middle, and last residues masked separately",
    "total_homologous_sequences": len(all_homologs),
    "mab_training_sequences": len(mab_train_seqs),
    "p5a_training_sequences": len(p5a_train_seqs),
    "cah2_training_sequences": len(cah2_train_seqs),
    "de_novo_cah2_sequences": len(de_novo_cah2_seqs),
    "mab_target_length": len(mab_target_seq),
    "cah2_target_length": len(cah2_target_seq),
    "masked_kmer_total_samples": int(X_ml.shape[0]),
    "training_samples": int(X_train_ml.shape[0]),
    "validation_samples": int(X_val_ml.shape[0]),
    "train_validation_split": "80:20 stratified split"
}

with open(
    os.path.join(METADATA_DIR, "dataset_generation_settings.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(settings, f, indent=4)

# Create ZIP file
zip_file = shutil.make_archive(
    base_name=GENERATED_DATA_DIR,
    format="zip",
    root_dir=GENERATED_DATA_DIR
)

print("\n==== Generated Data Saved Successfully ====")
print("Generated data folder:", GENERATED_DATA_DIR)
print("ZIP file:", zip_file)

print("\n==== Folder Structure ====")

for root, dirs, files in os.walk(GENERATED_DATA_DIR):
    level = root.replace(GENERATED_DATA_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)

    for file in files:
        print(f"{subindent}{file}")

==== Settings ====
FAST_MODE: False
MAX_ML_SAMPLES: 150000
KMER_SIZE: 11
TOP_N_MODELS: 3
BEAM_WIDTH: 5
RESIDUE_TOP_K: 5

==== Dataset Paths ====
DE_NOVO_CAH2 => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/de_novo_sequence_CAH2.txt | exists: True
MAB_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt | exists: True
MAB_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_training_sequence.txt | exists: True
P5A_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/p5A_training_sequence.txt | exists: True
CAH2_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/target_sequence_CAH2.txt | exists: True
CAH2_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/training_sequences_CAH2.txt | exists: True

==== Amino Acid Checks ====
Amino acids: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', '